# 06 — NER Evaluation with seqeval

**Goal:** Understand what the F1 number from notebook 5 actually measures, why **token-level** and **span-level** metrics differ, and how to do error analysis so you can improve the model.

## What you'll learn

1. Token-level vs span-level F1 — why the latter is what you report
2. Per-entity-type breakdown
3. Confusion between entity types
4. Reading the classification report
5. Common failure modes and how to spot them

## Step 1 — Token-level vs span-level: the key distinction

Suppose the gold label sequence is:

```
Lazarus  Group
B-ACTOR  I-ACTOR
```

And the model predicts:

```
Lazarus  Group
B-ACTOR  O
```

**Token-level F1**: 1 of 2 tokens correct → 50% accuracy on tokens.

**Span-level F1**: the entity `"Lazarus Group"` was not fully recovered → counted as a **complete miss** (0 correct spans out of 1).

Span-level is harsher, but it's what actually matters: you want complete entities, not fragments. `seqeval` computes span-level F1 by default.

## Step 2 — Load the model we trained in notebook 5

In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_dir = "./ner-cti-bert-final"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForTokenClassification.from_pretrained(model_dir)
model.eval()
id2label = model.config.id2label

## Step 3 — A held-out gold set

In [2]:
gold = [
    {"words": ["APT28", "uses", "X-Agent", "and", "Zebrocy", "."],
     "labels": ["B-THREAT_ACTOR", "O", "B-MALWARE", "O", "B-MALWARE", "O"]},
    {"words": ["Lazarus", "Group", "deployed", "WannaCry", "exploiting", "CVE-2017-0144", "."],
     "labels": ["B-THREAT_ACTOR", "I-THREAT_ACTOR", "O", "B-MALWARE", "O", "B-VULNERABILITY", "O"]},
    {"words": ["Emotet", "dropped", "TrickBot", "and", "Ryuk", "."],
     "labels": ["B-MALWARE", "O", "B-MALWARE", "O", "B-MALWARE", "O"]},
    {"words": ["CVE-2021-44228", "aka", "Log4Shell", "was", "widely", "exploited", "."],
     "labels": ["B-VULNERABILITY", "O", "O", "O", "O", "O", "O"]},
]

## Step 4 — Predict on gold set

In [3]:
def predict_labels(words):
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**enc).logits[0]
    pred_ids = logits.argmax(-1).tolist()
    wids = enc.word_ids()

    # collapse back to word-level: first subword's label
    word_labels = []
    seen = set()
    for wid, pid in zip(wids, pred_ids):
        if wid is None or wid in seen:
            continue
        word_labels.append(id2label[pid])
        seen.add(wid)
    return word_labels


y_true = [ex["labels"] for ex in gold]
y_pred = [predict_labels(ex["words"]) for ex in gold]

for ex, preds in zip(gold, y_pred):
    print(" ".join(ex["words"]))
    print("  gold:", ex["labels"])
    print("  pred:", preds)
    print()

APT28 uses X-Agent and Zebrocy .
  gold: ['B-THREAT_ACTOR', 'O', 'B-MALWARE', 'O', 'B-MALWARE', 'O']
  pred: ['B-THREAT_ACTOR', 'O', 'O', 'O', 'O', 'O']

Lazarus Group deployed WannaCry exploiting CVE-2017-0144 .
  gold: ['B-THREAT_ACTOR', 'I-THREAT_ACTOR', 'O', 'B-MALWARE', 'O', 'B-VULNERABILITY', 'O']
  pred: ['O', 'O', 'O', 'O', 'O', 'B-VULNERABILITY', 'O']

Emotet dropped TrickBot and Ryuk .
  gold: ['B-MALWARE', 'O', 'B-MALWARE', 'O', 'B-MALWARE', 'O']
  pred: ['B-THREAT_ACTOR', 'O', 'O', 'O', 'O', 'O']

CVE-2021-44228 aka Log4Shell was widely exploited .
  gold: ['B-VULNERABILITY', 'O', 'O', 'O', 'O', 'O', 'O']
  pred: ['B-VULNERABILITY', 'O', 'O', 'O', 'O', 'O', 'O']



## Step 5 — Overall span-level metrics

In [4]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

print(f"Precision (span): {precision_score(y_true, y_pred):.3f}")
print(f"Recall    (span): {recall_score(y_true, y_pred):.3f}")
print(f"F1        (span): {f1_score(y_true, y_pred):.3f}")

Precision (span): 0.750
Recall    (span): 0.300
F1        (span): 0.429


## Step 6 — Per-entity-type breakdown

This is the most actionable output. It tells you *which entity type is hurting you*.

In [5]:
print(classification_report(y_true, y_pred, digits=3))

               precision    recall  f1-score   support

      MALWARE      0.000     0.000     0.000         6
 THREAT_ACTOR      0.500     0.500     0.500         2
VULNERABILITY      1.000     1.000     1.000         2

    micro avg      0.750     0.300     0.429        10
    macro avg      0.500     0.500     0.500        10
 weighted avg      0.300     0.300     0.300        10



/home/saqib/venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


### How to read it

- **precision** = of the entities the model predicted, how many were correct → if low, model is **over-predicting**
- **recall** = of the real entities, how many were found → if low, model is **missing** entities
- **f1** = harmonic mean
- **support** = number of gold entities of that type

**Debugging heuristics:**
- Low precision, high recall → model labels too aggressively (see which types get over-tagged)
- High precision, low recall → model is too cautious (often due to class imbalance — few examples of that type)
- One type crashes to 0 → you probably have too few training examples of it, or its surface forms are too varied

## Step 7 — Error analysis: find wrong predictions

In [6]:
def diff_entities(words, gold_labels, pred_labels):
    """Return list of positions where gold and pred disagree."""
    return [
        (i, w, g, p)
        for i, (w, g, p) in enumerate(zip(words, gold_labels, pred_labels))
        if g != p
    ]


for ex, preds in zip(gold, y_pred):
    diffs = diff_entities(ex["words"], ex["labels"], preds)
    if not diffs:
        continue
    print("Sentence:", " ".join(ex["words"]))
    for i, w, g, p in diffs:
        print(f"  [{i}] {w!r} gold={g} pred={p}")
    print()

Sentence: APT28 uses X-Agent and Zebrocy .
  [2] 'X-Agent' gold=B-MALWARE pred=O
  [4] 'Zebrocy' gold=B-MALWARE pred=O

Sentence: Lazarus Group deployed WannaCry exploiting CVE-2017-0144 .
  [0] 'Lazarus' gold=B-THREAT_ACTOR pred=O
  [1] 'Group' gold=I-THREAT_ACTOR pred=O
  [3] 'WannaCry' gold=B-MALWARE pred=O

Sentence: Emotet dropped TrickBot and Ryuk .
  [0] 'Emotet' gold=B-MALWARE pred=B-THREAT_ACTOR
  [2] 'TrickBot' gold=B-MALWARE pred=O
  [4] 'Ryuk' gold=B-MALWARE pred=O



## Step 8 — Common failure modes to watch for

| Symptom | Likely cause | Fix |
|---|---|---|
| Entity missed entirely | Unseen surface form | More training data; domain model |
| Right type, wrong boundary (`Lazarus` tagged but not `Group`) | Subword alignment bug OR under-trained | Check alignment; more epochs |
| Wrong type (malware tagged as threat actor) | Ambiguous context, imbalanced training | Add disambiguating examples |
| `I-X` emitted with no preceding `B-X` | Normal model noise | `aggregation_strategy="simple"` cleans it up |
| Precision drops sharply at inference time | Distribution shift vs training data | Evaluate on more diverse held-out set |

## Your turn — exercises

1. Hand-build a 10-sentence gold set with entities your training data didn't cover. How does F1 change?
2. Write a function that takes `y_true`, `y_pred` and prints a **confusion count** — for each pair `(gold_type, pred_type)`, how many tokens were confused? (This is a token-level diagnostic, not span-level.)
3. Re-run notebook 5 training for **1 epoch** instead of 10. Re-evaluate here. Where do the F1 numbers drop the most?

## Next notebook

**`07_classification_data_prep.ipynb`** — we pivot from NER to sequence classification. You'll build a report-level labeled dataset, think about class balance, and prep it for training.